# 02. Text Preprocessing & Feature Engineering
## Support Ticket Classification & Prioritization

### Objective:
1. Inspect raw ticket text and identify linguistic noise (URLs, emails, punctuation, contractions).
2. Implement and test `clean_text` and the scikit-learn `TextCleanerTransformer`.
3. Explore TF-IDF vectorization parameters (`ngram_range`, `min_df`, `max_df`, `sublinear_tf`).
4. Extract top discriminating unigrams and bigrams per category.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import config
from src.data_loader import load_raw_data, check_missing_and_duplicates
from src.preprocessing import clean_text, TextCleanerTransformer
from src.feature_engineering import build_tfidf_vectorizer, extract_top_features_per_class

raw_df = load_raw_data()
cleaned_df, _ = check_missing_and_duplicates(raw_df)

### 1. Raw Text Inspection vs. Cleaned Text
Let us examine how `clean_text` normalizes noisy text while retaining technical tokens.

In [ ]:
sample_indices = [5, 42, 108]
for idx in sample_indices:
    raw = cleaned_df['text'].iloc[idx]
    cleaned = clean_text(raw)
    print(f'Ticket #{idx}:')
    print(f'  RAW    : {raw}')
    print(f'  CLEANED: {cleaned}\n')

### Analysis of Text Cleaning
- URLs (`https://...`) and email addresses are stripped without leaving orphan fragments.
- Contractions (`can't` -> `cannot`, `pls` -> `please`) are standardized.
- Key error numbers (`500`, `403`) are preserved because error codes are highly predictive of bugs and access issues.

### 2. TF-IDF Feature Extraction
We fit `TfidfVectorizer` to extract unigrams and bigrams with `min_df=2`, `max_df=0.85`, and `sublinear_tf=True`.

In [ ]:
cleaner = TextCleanerTransformer()
cleaned_texts = cleaner.transform(cleaned_df['text'])

vectorizer = build_tfidf_vectorizer()
X_tfidf = vectorizer.fit_transform(cleaned_texts)

print(f'TF-IDF Matrix Shape: {X_tfidf.shape}')
print(f'Extracted Vocabulary Size: {len(vectorizer.vocabulary_)} features')

### 3. Top Predictive Features per Category
We inspect the top 8 terms with the highest mean TF-IDF weight for each ticket category.

In [ ]:
top_features = extract_top_features_per_class(vectorizer, X_tfidf, cleaned_df['category'], top_n=8)
for cat, feats in top_features.items():
    print(f'\nCategory: [{cat}]')
    print('  Top Features:', ', '.join(feats))

### Analysis of Predictive Features
- **Bug / System Error**: Dominated by `crashes`, `error`, `unhandled exception`, `stack trace`, `nullpointerexception`.
- **Billing & Payment**: Dominated by `invoice`, `credit card`, `refund`, `subscription`, `declined`.
- **Account & Access**: Dominated by `password reset`, `2fa`, `okta`, `sso`, `locked out`.
- **Feature Request**: Dominated by `feature request`, `dark mode`, `webhook`, `export`, `ability`.
- **Technical / IT Support**: Dominated by `vpn`, `wi-fi`, `docking station`, `outlook`, `monitor`.
The extracted n-grams map directly to the domain concepts, confirming that classical TF-IDF provides strong signal.

### Summary

### Q&A
- **Q: Why use bigrams (ngram_range=(1, 2))?**
  - **A**: Unigrams like 'credit' or 'dark' lose meaning; bigrams like 'credit card' and 'dark mode' carry distinct category signals.
- **Q: Why use sublinear_tf?**
  - **A**: Sublinear scaling prevents tickets with repeated keywords from disproportionately skewing the dot products in linear classifiers.

### Data Analysis Key Findings
- Preprocessing reduced unique token count while preserving technical error identifiers.
- TF-IDF matrix has 2,165 rows and a compact, high-signal vocabulary of ~2,000 features.

### Insights or Next Steps
- Embed `TextCleanerTransformer` and `TfidfVectorizer` inside an end-to-end scikit-learn `Pipeline` to benchmark candidate models.